In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window


def clean_price_data(df: DataFrame):
    """
    Clean raw electricity price data for downstream feature engineering.

    This transformation:
    - casts `measured_at` to a timestamp column named `timestamp_utc`
    - converts `eur_per_mwh` from a comma-decimal string to a double
    - drops the original `measured_at` column

    Args:
        df: Spark DataFrame containing raw price data with at least
            `measured_at` and `eur_per_mwh` columns.

    Returns:
        A Spark DataFrame with:
        - `timestamp_utc` as timestamp
        - `eur_per_mwh` as double
    """
    return (
        df
        .withColumn("timestamp_utc", F.col("measured_at").cast("timestamp"))
        .withColumn(
            "eur_per_mwh",
            F.regexp_replace("eur_per_mwh", ",", ".").cast("double")
        )
        .drop("measured_at")
    )


def create_price_lag_features(df: DataFrame):
    """
    Create lag-based price features for forecasting.

    The function generates hourly lag features from `eur_per_mwh`:
    - `price_lag_24`: price 24 hours earlier
    - `price_lag_48`: price 48 hours earlier
    - `price_lag_168`: price 168 hours earlier (7 days earlier)

    After the lag features are created, the current `eur_per_mwh` column
    is dropped so that only lagged values remain. This is useful when
    future raw price values are not available at inference time.

    Args:
        df: Spark DataFrame containing `timestamp_utc` and `eur_per_mwh`.

    Returns:
        A Spark DataFrame with lag feature columns and without
        the raw `eur_per_mwh` column.
    """
    window = Window.orderBy("timestamp_utc")

    return (
        df
        .withColumn("price_lag_24", F.lag("eur_per_mwh", 24).over(window))
        .withColumn("price_lag_48", F.lag("eur_per_mwh", 48).over(window))
        .withColumn("price_lag_168", F.lag("eur_per_mwh", 168).over(window))
        .drop("eur_per_mwh")
    )


def drop_initial_lag_rows(df: DataFrame, min_history_rows: int = 336):
    """
    Drop the earliest rows that do not have sufficient historical context.

    Lag features introduce null values at the beginning of the time series.
    This function assigns a row number ordered by `timestamp_utc` and removes
    the first `min_history_rows` rows.

    Args:
        df: Spark DataFrame ordered logically by `timestamp_utc`.
        min_history_rows: Number of initial rows to remove. Defaults to 336.

    Returns:
        A Spark DataFrame with the first `min_history_rows` rows removed.
    """
    window = Window.orderBy("timestamp_utc")

    return (
        df
        .withColumn("row_num", F.row_number().over(window))
        .filter(F.col("row_num") > min_history_rows)
        .drop("row_num")
    )


def split_price_training_and_inference_data(df: DataFrame, cutoff_start: str, cutoff_end: str):
    """
    Split price feature data into training and inference datasets.

    The training dataset contains rows earlier than `cutoff_start`.
    The inference dataset contains rows in the half-open interval
    [`cutoff_start`, `cutoff_end`).

    Args:
        df: Spark DataFrame containing `timestamp_utc`.
        cutoff_start: Inclusive start timestamp for the inference window.
        cutoff_end: Exclusive end timestamp for the inference window.

    Returns:
        A tuple of:
        - training DataFrame
        - inference DataFrame
    """
    training_df = df.filter(
        F.col("timestamp_utc") < F.lit(cutoff_start).cast("timestamp")
    )

    inference_df = df.filter(
        (F.col("timestamp_utc") >= F.lit(cutoff_start).cast("timestamp")) &
        (F.col("timestamp_utc") < F.lit(cutoff_end).cast("timestamp"))
    )

    return training_df, inference_df


def write_dataframe_to_table(df: DataFrame, destination: str):
    """
    Write a Spark DataFrame to a managed table using overwrite mode.

    Args:
        df: Spark DataFrame to write.
        destination: Fully qualified destination table name.

    Returns:
        None.

    Side Effects:
        Overwrites the destination table in the metastore.
    """
    df.write.mode("overwrite").saveAsTable(destination)


if __name__ == "__main__":
    catalog = "fortum_challenge_data"
    bronze_schema = "01_bronze"
    silver_schema = "02_silver"

    raw_table = "price_data_raw"
    training_table = "price_training_clean"
    inference_table = "price_inference_clean"

    source = f"{catalog}.{bronze_schema}.{raw_table}"
    training_destination = f"{catalog}.{silver_schema}.{training_table}"
    inference_destination = f"{catalog}.{silver_schema}.{inference_table}"

    cutoff_start = "2024-09-29 00:00:00"
    cutoff_end = "2024-10-01 00:00:00"

    df_raw = spark.table(source)

    df_price_clean = clean_price_data(df_raw)
    df_price_features = create_price_lag_features(df_price_clean)
    df_price_ready = drop_initial_lag_rows(df_price_features, min_history_rows=336)

    df_train, df_inference = split_price_training_and_inference_data(
        df_price_ready,
        cutoff_start=cutoff_start,
        cutoff_end=cutoff_end,
    )

    write_dataframe_to_table(df_train, training_destination)
    write_dataframe_to_table(df_inference, inference_destination)

    df_inference.display()